# Flight Delay Intelligence System — Preprocessing
This notebook cleans the raw working dataset and prepares it for EDA and modeling.

In [79]:
import pandas as pd
import numpy as np

## 1. Load Data
Loading flights_working.csv — 3M sampled flights, 2014–2018.

In [ ]:
df=pd.read_csv(r"data/raw/flights_working.csv")

In [81]:
df.head(3)

,FL_DATE,OP_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,DEST,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,TAXI_OUT,WHEELS_OFF,...,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,Unnamed: 27
0,2014-06-14,F9,407,DEN,LAX,1720,1826.0,66.0,10.0,1836.0,...,140.0,140.0,120.0,862.0,0.0,0.0,32.0,0.0,34.0,NaN
1,2014-07-21,WN,4704,CMH,BWI,1555,1555.0,0.0,16.0,1611.0,...,75.0,76.0,55.0,337.0,NaN,NaN,NaN,NaN,NaN,NaN
2,2014-10-09,AA,43,DTW,DFW,1450,1456.0,6.0,11.0,1507.0,...,175.0,171.0,141.0,986.0,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Drop Empty Column
`Unnamed: 27` is a 100% null trailing artifact from the original CSV. Dropped.

In [82]:
df.drop('Unnamed: 27',axis=1,inplace=True)

In [83]:
df.shape

(3000000, 27)

## 3. Parse Dates
Converting `FL_DATE` from string to datetime and extracting `YEAR`, `MONTH`, `DAY`, `DAY_OF_WEEK`. Dropping `FL_DATE` after extraction.

In [84]:
df['FL_DATE']=pd.to_datetime(df['FL_DATE'])

In [85]:
df['MONTH']=df['FL_DATE'].dt.month
df['YEAR']=df['FL_DATE'].dt.year
df['DAY']=df['FL_DATE'].dt.day
df['DAY_OF_WEEK']=df['FL_DATE'].dt.day_of_week

In [86]:
df.drop('FL_DATE',axis=1,inplace=True)

## 4. Convert HHMM to Hour
`CRS_DEP_TIME` and `CRS_ARR_TIME` are stored as HHMM integers (e.g. 1430 = 2:30 PM). Converting to hour only (e.g. 14) for model compatibility.

In [87]:
df['CRS_DEP_TIME']=df['CRS_DEP_TIME']//100
df['CRS_ARR_TIME']=df['CRS_ARR_TIME']//100

In [88]:
df.head(3)

,OP_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,DEST,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,TAXI_OUT,WHEELS_OFF,WHEELS_ON,...,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,MONTH,YEAR,DAY,DAY_OF_WEEK
0,F9,407,DEN,LAX,17,1826.0,66.0,10.0,1836.0,1936.0,...,862.0,0.0,0.0,32.0,0.0,34.0,6,2014,14,5
1,WN,4704,CMH,BWI,15,1555.0,0.0,16.0,1611.0,1706.0,...,337.0,NaN,NaN,NaN,NaN,NaN,7,2014,21,0
2,AA,43,DTW,DFW,14,1456.0,6.0,11.0,1507.0,1628.0,...,986.0,NaN,NaN,NaN,NaN,NaN,10,2014,9,3


## 5. Separate Cancelled Flights
Cancelled flights (1.6%, ~47,949 rows) are separated into `df_cancelled` for future cancellation analysis. Removed from main `df` since they have no arrival delay — not relevant for delay prediction.

In [89]:
df_cancelled=df[df['CANCELLED']==1]

In [90]:
df_cancelled

,OP_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,DEST,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,TAXI_OUT,WHEELS_OFF,WHEELS_ON,...,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,MONTH,YEAR,DAY,DAY_OF_WEEK
20,WN,1026,LAS,SAN,12,NaN,NaN,NaN,NaN,NaN,...,258.0,NaN,NaN,NaN,NaN,NaN,7,2014,16,2
62,EV,4408,IAH,OMA,18,NaN,NaN,NaN,NaN,NaN,...,781.0,NaN,NaN,NaN,NaN,NaN,7,2014,7,0
114,B6,2701,BUF,JFK,19,NaN,NaN,NaN,NaN,NaN,...,301.0,NaN,NaN,NaN,NaN,NaN,6,2014,11,2
192,EV,4201,DAY,EWR,19,NaN,NaN,NaN,NaN,NaN,...,533.0,NaN,NaN,NaN,NaN,NaN,4,2014,10,3
193,US,435,ATL,PHX,7,NaN,NaN,NaN,NaN,NaN,...,1587.0,NaN,NaN,NaN,NaN,NaN,2,2014,13,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2999790,OO,4168,LGA,ORD,20,NaN,NaN,NaN,NaN,NaN,...,733.0,NaN,NaN,NaN,NaN,NaN,11,2018,26,0
2999840,OO,5252,MFR,SFO,6,NaN,NaN,NaN,NaN,NaN,...,329.0,NaN,NaN,NaN,NaN,NaN,12,2018,21,4
2999865,OO,3882,GSO,LGA,18,NaN,NaN,NaN,NaN,NaN,...,461.0,NaN,NaN,NaN,NaN,NaN,12,2018,9,6
2999934,AA,1105,JFK,BOS,21,2156.0,36.0,59.0,2255.0,NaN,...,187.0,NaN,NaN,NaN,NaN,NaN,1,2018,5,4


In [91]:
df=df[df['CANCELLED']==0]

## 6. Drop Irrelevant Columns
Dropping `CANCELLED`, `CANCELLATION_CODE`, and `DIVERTED` — no longer needed for delay prediction model.

In [92]:
df=df.drop(columns=['CANCELLED','CANCELLATION_CODE','DIVERTED'])

In [93]:
df.shape

(2952051, 27)

## 7. Handle Invalid Elapsed Time
Dropping rows where `CRS_ELAPSED_TIME` is null or negative — physically impossible values, negligible count (2 rows).

In [94]:
df[(df['CRS_ELAPSED_TIME'] < 0) | (df['CRS_ELAPSED_TIME'].isnull())]

,OP_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,DEST,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,TAXI_OUT,WHEELS_OFF,WHEELS_ON,...,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,MONTH,YEAR,DAY,DAY_OF_WEEK
2675838,YX,3650,IAH,SDF,19,1942.0,-3.0,17.0,1959.0,4.0,...,788.0,NaN,NaN,NaN,NaN,NaN,2,2018,24,5
2719461,YX,3480,EWR,RDU,6,613.0,-7.0,25.0,638.0,1130.0,...,416.0,NaN,NaN,NaN,NaN,NaN,2,2018,20,1


In [95]:
df = df.dropna(subset=['CRS_ELAPSED_TIME'])
df = df[df['CRS_ELAPSED_TIME'] > 0]

In [96]:
df.shape

(2952049, 27)

## 8. Create Target Variable
Creating `ARR_DEL15` — binary column indicating whether a flight was delayed by 15+ minutes on arrival. This is the official BTS definition of a delay and will be the model's prediction target.

Class distribution: ~82% on-time (0), ~18% delayed (1). Mild imbalance — will evaluate using F1/precision/recall rather than accuracy.

In [97]:
df['ARR_DEL15']=np.where(df['ARR_DELAY']>=15,1,0)

In [98]:
df.sample()

,OP_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,DEST,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,TAXI_OUT,WHEELS_OFF,WHEELS_ON,...,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,MONTH,YEAR,DAY,DAY_OF_WEEK,ARR_DEL15
825859,OO,6527,SLC,DEN,19,1921.0,-9.0,21.0,1942.0,2041.0,...,NaN,NaN,NaN,NaN,NaN,1,2015,22,3,0


In [99]:
df['ARR_DEL15'].value_counts()

ARR_DEL15
0    2389657
1     562392
Name: count, dtype: int64

## 9. Fill Delay Cause Columns
Filling nulls in `CARRIER_DELAY`, `WEATHER_DELAY`, `NAS_DELAY`, `SECURITY_DELAY`, `LATE_AIRCRAFT_DELAY` with 0. Null here means no delay occurred — 0 is logically correct, not an assumption.

In [101]:
df[['CARRIER_DELAY','WEATHER_DELAY','NAS_DELAY','SECURITY_DELAY','LATE_AIRCRAFT_DELAY']]=df[['CARRIER_DELAY','WEATHER_DELAY','NAS_DELAY','SECURITY_DELAY','LATE_AIRCRAFT_DELAY']].fillna(0)

In [102]:
df.isnull().sum()

OP_CARRIER                0
OP_CARRIER_FL_NUM         0
ORIGIN                    0
DEST                      0
CRS_DEP_TIME              0
DEP_TIME                  0
DEP_DELAY               389
TAXI_OUT                  0
WHEELS_OFF                0
WHEELS_ON              1191
TAXI_IN                1191
CRS_ARR_TIME              0
ARR_TIME               1191
ARR_DELAY              7588
CRS_ELAPSED_TIME          0
ACTUAL_ELAPSED_TIME    7396
AIR_TIME               7396
DISTANCE                  0
CARRIER_DELAY             0
WEATHER_DELAY             0
NAS_DELAY                 0
SECURITY_DELAY            0
LATE_AIRCRAFT_DELAY       0
MONTH                     0
YEAR                      0
DAY                       0
DAY_OF_WEEK               0
ARR_DEL15                 0
dtype: int64

## 10. Drop Null Arrival Records
Dropping rows where `ARR_DELAY` is null — these are diverted flights that never arrived at destination. No arrival = no delay value = unusable for prediction.

In [103]:
df=df.dropna(subset=['ARR_DELAY'])

In [104]:
df.shape

(2944461, 28)

In [105]:
df.isnull().sum()

OP_CARRIER               0
OP_CARRIER_FL_NUM        0
ORIGIN                   0
DEST                     0
CRS_DEP_TIME             0
DEP_TIME                 0
DEP_DELAY              383
TAXI_OUT                 0
WHEELS_OFF               0
WHEELS_ON                0
TAXI_IN                  0
CRS_ARR_TIME             0
ARR_TIME                 0
ARR_DELAY                0
CRS_ELAPSED_TIME         0
ACTUAL_ELAPSED_TIME      0
AIR_TIME                 0
DISTANCE                 0
CARRIER_DELAY            0
WEATHER_DELAY            0
NAS_DELAY                0
SECURITY_DELAY           0
LATE_AIRCRAFT_DELAY      0
MONTH                    0
YEAR                     0
DAY                      0
DAY_OF_WEEK              0
ARR_DEL15                0
dtype: int64

## 11. Drop Null Departure Records
Dropping 383 rows with null `DEP_DELAY` — edge case with arrival record but no departure record. Negligible count.

In [106]:
df = df.dropna(subset=['DEP_DELAY'])

In [107]:
df.shape

(2944078, 28)

## 12. Save Processed Dataset
Saving cleaned dataframe to `data/processed/flights_processed.csv`.

In [108]:
df.to_csv("data/processed/flights_processed.csv", index=False)

## 13. Preprocessing Summary

| Step | Action | Rows Affected |
|---|---|---|
| Drop Unnamed column | Removed 1 column | — |
| Parse dates | Extracted YEAR, MONTH, DAY, DAY_OF_WEEK | — |
| Convert time columns | HHMM → hour | — |
| Remove cancelled flights | Saved to df_cancelled | -47,949 |
| Drop irrelevant columns | CANCELLED, CANCELLATION_CODE, DIVERTED | — |
| Handle invalid elapsed time | Dropped null/negative rows | -2 |
| Fill delay cause nulls | Filled with 0 | — |
| Drop null ARR_DELAY | Diverted flights removed | -7,588 |
| Drop null DEP_DELAY | Edge case rows removed | -383 |
| **Final dataset** | **Ready for EDA** | **2,944,078 rows** |
